In [39]:
import pandas as pd 
import numpy as np

In [40]:
df = pd.read_csv("creditcard.csv")

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
print("All libraries are successful!")

All libraries are successful!


In [42]:
def check_data_quality(df):
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

check_data_quality(df)

Missing values: 0
Duplicate rows: 1081


In [43]:
def clean_dataset(df):
    df = df.copy()

    duplicate_count = df.duplicated().sum()
    print("Duplicate rows found:", duplicate_count)

    df = df.drop_duplicates()

    print("Shape after cleaning:", df.shape)

    return df

df_clean = clean_dataset(df)

Duplicate rows found: 1081
Shape after cleaning: (283726, 31)


In [44]:
def select_features(df, target_column="Class"):

    feature_columns = [col for col in df.columns if col != target_column]
    model_df = df[feature_columns + [target_column]].copy()

    return model_df, feature_columns

model_df, feature_columns = select_features(df)

In [45]:
def clean_target(model_df, target_column="Class"):

    model_df = model_df.copy()

    model_df[target_column] = pd.to_numeric(
        model_df[target_column], errors="coerce"
    )

    model_df = model_df.dropna(subset=[target_column])

    return model_df

model_df = clean_target(model_df)

In [55]:
def separate_input_target(model_df, feature_columns, target_column="Class"):

    X = model_df[feature_columns]
    y = model_df[target_column]

    return X, y

In [47]:
def identify_features(X):
   numerical_features = X.select_dtypes( include=np.number).columns.tolist()
   categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
   
   print("Numerical features:", len(numerical_features))
   print("Categorical features:", len(categorical_features))
   
   return numerical_features, categorical_features

numerical_features, categorical_features = identify_features(X)

Numerical features: 30
Categorical features: 0


In [48]:
def create_numeric_pipeline():

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
        ])

    return numeric_transformer

In [49]:
def create_categorical_pipeline():

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return categorical_transformer



In [50]:
def create_preprocessor(X):

    numeric_features, categorical_features = identify_features(X)

    transformers = []

    if numeric_features:
        transformers.append(
            ("numeric", create_numeric_pipeline(), numeric_features)
        )

    if categorical_features:
        transformers.append(
            ("categorical", create_categorical_pipeline(), categorical_features)
        )

    preprocessor = ColumnTransformer(transformers=transformers)

    return preprocessor

preprocessor = create_preprocessor(X)

Numerical features: 30
Categorical features: 0


In [51]:
def preprocess_dataset(df, target_column="Class"):

    df_clean = clean_dataset(df)

    model_df, feature_columns = select_features(df_clean, target_column)

    model_df = clean_target(model_df, target_column)

    X, y = separate_input_target(model_df, feature_columns, target_column)

    preprocessor = create_preprocessor(X)

    return X, y, preprocessor


In [56]:
X, y, preprocessor = preprocess_dataset(df)


Duplicate rows found: 1081
Shape after cleaning: (283726, 31)
Numerical features: 30
Categorical features: 0


In [57]:
#Final results of cleaned datasets
print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nClass distribution:")
print(y.value_counts())

print("\nFinally!, Preprocessing pipeline created successfully!")

Input shape: (283726, 30)
Target shape: (283726,)
Numerical features: 30
Categorical features: 0

Class distribution:
Class
0    283253
1       473
Name: count, dtype: int64

Finally!, Preprocessing pipeline created successfully!
